# OncoBridge — Stage 1 Component Ablation (S1.4), Full Convergence

This notebook reruns the Stage 1 component sweep (gene-importance gating, cross-attention depth, attention heads) **at the same full training budget as the main model** (140 epochs, patience 18) — not the reduced 25-epoch budget used for the earlier internal-comparison-only sweep.

Data loading, gene selection, and the model architecture are copied unchanged from `oncobridge-mmcat-methylation-id-genemapping...ipynb` (the v7 training notebook). The only thing this notebook adds is a config-driven switch that turns one architecture component off/down at a time.

**To run an ablation:** set `ABLATION` in the cell below, then Restart & Run All. Each run saves its own checkpoint and appends one row to a results CSV, so running all four options across separate kernel sessions builds up one comparison table.

**Options:**
- `'full'` — reference run, all components at their normal setting (sanity check against the 95.74% headline; optional, only needed if you want a fresh full-budget confirmation)
- `'no_gene_gate'` — GeneImportanceLayer disabled (each modality encoder skips the per-gene sigmoid gate)
- `'shallow_cross_attn'` — cross-modal attention depth reduced from 4 layers to 1
- `'few_heads'` — attention heads reduced from 8 to 2 (applies to both per-modality encoders and cross-modal attention, matching how `num_heads` is used in the original architecture)


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.utils.checkpoint import checkpoint as grad_checkpoint
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler, MaxAbsScaler
from sklearn.metrics import (
    classification_report, accuracy_score, f1_score, matthews_corrcoef, confusion_matrix
)
from sklearn.utils.class_weight import compute_class_weight
from sklearn.feature_selection import VarianceThreshold
import copy, time, warnings, math, os, csv
from datetime import datetime
warnings.filterwarnings('ignore')

## ⚙️ Ablation selector — edit this cell only, then Restart & Run All

In [2]:
# ══════════════════════════════════════════════════════════════════════════
#  ABLATION SELECTOR — the only thing you need to change between runs
# ══════════════════════════════════════════════════════════════════════════
ABLATION = 'no_gene_gate'   # 'full' | 'no_gene_gate' | 'shallow_cross_attn' | 'few_heads'

RESULTS_LOG = '/kaggle/working/oncobridge_s14_full_convergence_results.csv'

# ── Base config — identical to the main v7 training run ────────────────────
CONFIG = {
    # Data paths (same as the Stage 1 training notebook)
    'mrna_path'  : '/kaggle/input/datasets/proutkarshtiwari/multi-omnics-data-muationmrnacnvmethyl-idgene/mrna_final.parquet',
    'cnv_path'   : '/kaggle/input/datasets/proutkarshtiwari/multi-omnics-data-muationmrnacnvmethyl-idgene/cnv_final.parquet',
    'mut_path'   : '/kaggle/input/datasets/proutkarshtiwari/multi-omnics-data-muationmrnacnvmethyl-idgene/mut_final.parquet',
    'meth_path'  : '/kaggle/input/datasets/proutkarshtiwari/multi-omnics-data-muationmrnacnvmethyl-idgene/meth_final.parquet',
    'label_path' : '/kaggle/input/datasets/proutkarshtiwari/multi-omnics-data-muationmrnacnvmethyl-idgene/labels_final.parquet',

    # Gene selection (unchanged from main run)
    'auto_k_coverage': 0.88,
    'mrna_max_k' : 9000,
    'cnv_max_k'  : 3500,
    'mut_max_k'  : 2500,
    'meth_max_k' : 6000,

    # Architecture defaults (proven v7 values — ablation overrides applied below)
    'embed_dim'         : 384,
    'num_heads'         : 8,
    'num_encoder_layers': 6,
    'num_cross_layers'  : 4,
    'cnn_kernel'        : 16,
    'cnn_stride'        : 16,
    'ff_dim'            : 1536,
    'dropout'           : 0.20,
    'gate_init'         : 1.5,
    'use_gene_gate'     : True,

    # Training — FULL BUDGET, same as the main model (this is the point of this notebook)
    'epochs'     : 140,
    'batch_size' : 64,
    'grad_accum' : 8,
    'lr'         : 2e-4,
    'weight_decay': 3e-4,
    'patience'   : 18,
    'label_smoothing': 0.05,
    'clip_grad'  : 1.0,
    'warmup_frac': 0.15,
    'num_workers': 4,
    'use_amp'    : True,
    'use_grad_ckpt': True,

    'max_class_weight': 8.0,

    'use_mixup'  : True,
    'mixup_prob' : 0.50,
    'mixup_alpha': 0.2,

    'balanced_classes_only': False,
    'min_samples_per_class': 100,

    'use_test_set': True,
    'seed'        : 42,
}

# ── Ablation-specific overrides ─────────────────────────────────────────────
ABLATION_OVERRIDES = {
    'full':               {},
    'no_gene_gate':       {'use_gene_gate': False},
    'shallow_cross_attn': {'num_cross_layers': 1},
    'few_heads':          {'num_heads': 2},
}

if ABLATION not in ABLATION_OVERRIDES:
    raise ValueError(f"Unknown ABLATION '{ABLATION}'. Choose one of {list(ABLATION_OVERRIDES)}")

CONFIG.update(ABLATION_OVERRIDES[ABLATION])
CONFIG['ablation_name'] = ABLATION
CONFIG['checkpoint'] = f'oncobridge_s14_{ABLATION}_fullconv.pt'

print(f'Selected ablation: {ABLATION}')
print(f'Overrides applied: {ABLATION_OVERRIDES[ABLATION] or "(none — reference full-model run)"}')
print(f'Checkpoint will be saved to: {CONFIG["checkpoint"]}')
print(f'Results will be appended to: {RESULTS_LOG}')

Selected ablation: no_gene_gate
Overrides applied: {'use_gene_gate': False}
Checkpoint will be saved to: oncobridge_s14_no_gene_gate_fullconv.pt
Results will be appended to: /kaggle/working/oncobridge_s14_full_convergence_results.csv


## Hardware setup

In [3]:
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
N_GPUS = torch.cuda.device_count() if torch.cuda.is_available() else 0

if torch.cuda.is_available():
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32       = True
    torch.backends.cudnn.benchmark        = True
    for i in range(N_GPUS):
        gb = torch.cuda.get_device_properties(i).total_memory / 1e9
        print(f'  GPU {i}: {torch.cuda.get_device_name(i)} — {gb:.1f} GB')

print(f'Primary device: {DEVICE}')
print(f'DataParallel: {N_GPUS > 1} ({N_GPUS} GPUs)')
print(f'Ablation: {ABLATION} | Effective batch: {CONFIG["batch_size"]} x {CONFIG["grad_accum"]} = {CONFIG["batch_size"]*CONFIG["grad_accum"]}')

  GPU 0: Tesla T4 — 15.6 GB
  GPU 1: Tesla T4 — 15.6 GB
Primary device: cuda
DataParallel: True (2 GPUs)
Ablation: no_gene_gate | Effective batch: 64 x 8 = 512


## Data loading — identical to the main Stage 1 notebook

In [4]:
print('Loading 4 modalities from parquet...')
t0 = time.time()

mrna_df  = pd.read_parquet(CONFIG['mrna_path'])
cnv_df   = pd.read_parquet(CONFIG['cnv_path'])
mut_df   = pd.read_parquet(CONFIG['mut_path'])
meth_df  = pd.read_parquet(CONFIG['meth_path'])
label_df = pd.read_parquet(CONFIG['label_path'])

print(f'  mRNA:        {mrna_df.shape}')
print(f'  CNV:         {cnv_df.shape}')
print(f'  Mutation:    {mut_df.shape}')
print(f'  Methylation: {meth_df.shape}')
print(f'  Labels:      {label_df.shape}')
print(f'  Loaded in {time.time()-t0:.1f}s')

le = LabelEncoder()
y  = le.fit_transform(label_df['_primary_disease'])
CONFIG['num_classes'] = len(le.classes_)

X = {
    'mrna': mrna_df.values.astype(np.float32),
    'cnv' : cnv_df.values.astype(np.float32),
    'mut' : mut_df.values.astype(np.float32),
    'meth': meth_df.values.astype(np.float32),
}
del mrna_df, cnv_df, mut_df, meth_df
import gc; gc.collect()

print(f'\nClasses: {CONFIG["num_classes"]}')

Loading 4 modalities from parquet...
  mRNA:        (7912, 15816)
  CNV:         (7912, 15816)
  Mutation:    (7912, 15816)
  Methylation: (7912, 15816)
  Labels:      (7912, 1)
  Loaded in 17.4s

Classes: 32


## Class filter and train / val / test split — identical to the main notebook

In [5]:
if CONFIG['balanced_classes_only']:
    min_n   = CONFIG['min_samples_per_class']
    counts  = np.bincount(y)
    keep    = np.where(counts >= min_n)[0]
    mask    = np.isin(y, keep)
    for m in X: X[m] = X[m][mask]
    le_new  = LabelEncoder()
    y       = le_new.fit_transform(y[mask])
    le.classes_ = le.classes_[keep]
    CONFIG['num_classes'] = len(le.classes_)
    print(f'Kept {CONFIG["num_classes"]} classes (>={min_n} samples)')
else:
    print(f'Using ALL {CONFIG["num_classes"]} classes')

print(f'Total samples: {len(y)}')

idx = np.arange(len(y))
if CONFIG['use_test_set']:
    tr_idx, tmp = train_test_split(idx, test_size=0.30, stratify=y, random_state=CONFIG['seed'])
    vl_idx, te_idx = train_test_split(tmp, test_size=0.50, stratify=y[tmp], random_state=CONFIG['seed'])
    print(f'Split: Train={len(tr_idx)} | Val={len(vl_idx)} | Test={len(te_idx)}')
else:
    tr_idx, vl_idx = train_test_split(idx, test_size=0.20, stratify=y, random_state=CONFIG['seed'])
    te_idx = None
    print(f'Split: Train={len(tr_idx)} | Val={len(vl_idx)}')

Using ALL 32 classes
Total samples: 7912
Split: Train=5538 | Val=1187 | Test=1187


## Gene selection (variance-coverage auto-k) — identical to the main notebook

In [6]:
MAX_K = {'mrna': CONFIG['mrna_max_k'], 'cnv': CONFIG['cnv_max_k'],
         'mut': CONFIG['mut_max_k'], 'meth': CONFIG['meth_max_k']}
SCALER_TYPE = {'mrna': 'standard', 'cnv': 'maxabs', 'mut': 'none', 'meth': 'none'}

def select_and_scale(X_full, split_indices, coverage, max_k, scaler_type, tag):
    X_tr_raw = X_full[split_indices[0]]
    X_vl_raw = X_full[split_indices[1]]
    X_te_raw = X_full[split_indices[2]] if split_indices[2] is not None else None

    vt = VarianceThreshold(threshold=0.0)
    vt.fit(X_tr_raw)
    variances = vt.variances_

    sorted_var = np.sort(variances)[::-1]
    total_var  = sorted_var.sum()
    cumsum     = np.cumsum(sorted_var)
    auto_k     = int(np.searchsorted(cumsum, coverage * total_var)) + 1
    k          = min(auto_k, max_k)

    top_idx = np.argsort(variances)[::-1][:k]
    X_tr = X_tr_raw[:, top_idx]
    X_vl = X_vl_raw[:, top_idx]
    X_te = X_te_raw[:, top_idx] if X_te_raw is not None else None

    if scaler_type == 'standard':
        sc = StandardScaler()
        X_tr = sc.fit_transform(X_tr).astype(np.float32)
        X_vl = sc.transform(X_vl).astype(np.float32)
        X_te = sc.transform(X_te).astype(np.float32) if X_te is not None else None
    elif scaler_type == 'maxabs':
        sc = MaxAbsScaler()
        X_tr = sc.fit_transform(X_tr).astype(np.float32)
        X_vl = sc.transform(X_vl).astype(np.float32)
        X_te = sc.transform(X_te).astype(np.float32) if X_te is not None else None
    else:
        X_tr = X_tr.astype(np.float32)
        X_vl = X_vl.astype(np.float32)
        X_te = X_te.astype(np.float32) if X_te is not None else None

    print(f'  [{tag:4s}] auto-k={auto_k} -> capped at {k}')
    return X_tr, X_vl, X_te, k

print(f'=== Gene Selection (variance coverage={CONFIG["auto_k_coverage"]*100:.0f}%) ===')
splits = (tr_idx, vl_idx, te_idx)
Xp, gene_k = {}, {}
for mod in ['mrna', 'cnv', 'mut', 'meth']:
    tr, vl, te, k = select_and_scale(X[mod], splits, CONFIG['auto_k_coverage'],
                                      MAX_K[mod], SCALER_TYPE[mod], mod)
    Xp[mod] = (tr, vl, te)
    gene_k[mod] = k
    CONFIG[f'num_{mod}_genes'] = k

print(f'\nSelected genes: {gene_k}')
del X; gc.collect()

cw  = compute_class_weight('balanced', classes=np.unique(y[tr_idx]), y=y[tr_idx])
cw  = np.clip(cw, None, CONFIG['max_class_weight'])
cw_tensor = torch.FloatTensor(cw).to(DEVICE)
print(f'Class weights: {cw.min():.3f} - {cw.max():.3f}')

=== Gene Selection (variance coverage=88%) ===
  [mrna] auto-k=8069 -> capped at 8069
  [cnv ] auto-k=13433 -> capped at 3500
  [mut ] auto-k=9925 -> capped at 2500
  [meth] auto-k=7511 -> capped at 6000

Selected genes: {'mrna': 8069, 'cnv': 3500, 'mut': 2500, 'meth': 6000}
Class weights: 0.383 - 8.000


## Dataset / DataLoader — identical to the main notebook

In [7]:
class MultiOmicsDataset(Dataset):
    def __init__(self, mrna, cnv, mut, meth, labels):
        self.mrna   = torch.FloatTensor(mrna)
        self.cnv    = torch.FloatTensor(cnv)
        self.mut    = torch.FloatTensor(mut)
        self.meth   = torch.FloatTensor(meth)
        self.labels = torch.LongTensor(labels)

    def __len__(self): return len(self.labels)

    def __getitem__(self, i):
        return self.mrna[i], self.cnv[i], self.mut[i], self.meth[i], self.labels[i]


def make_loader(split_key, shuffle):
    key_map = {'tr': 0, 'vl': 1, 'te': 2}
    ki = key_map[split_key]
    idx_map = {'tr': tr_idx, 'vl': vl_idx, 'te': te_idx}
    idx = idx_map[split_key]

    def get(mod):
        arr = Xp[mod][ki]
        if arr is None: arr = Xp[mod][1]
        return arr

    ds = MultiOmicsDataset(get('mrna'), get('cnv'), get('mut'), get('meth'), y[idx])
    nw = CONFIG['num_workers']
    return DataLoader(ds, batch_size=CONFIG['batch_size'], shuffle=shuffle,
                       num_workers=nw, pin_memory=True,
                       persistent_workers=(nw > 0),
                       prefetch_factor=2 if nw > 0 else None,
                       drop_last=shuffle)

train_loader = make_loader('tr', shuffle=True)
val_loader   = make_loader('vl', shuffle=False)
test_loader  = make_loader('te', shuffle=False) if te_idx is not None else None
print(f'Train batches: {len(train_loader)} | Val batches: {len(val_loader)}')

Train batches: 86 | Val batches: 19


## Architecture

Same as `OncoBridgeMMCAT_v7`, with two additions so the ablations are a config switch rather than separate code:

- `ModalityEncoder` accepts `use_gene_gate`; when `False` it skips the `GeneImportanceLayer` entirely (identity on the input) instead of applying it.
- `num_cross_layers` and `num_heads` are read from `CONFIG` exactly as in the original — reducing them in the config cell above is all that's needed for `shallow_cross_attn` and `few_heads`.

No other part of the architecture changes.

In [8]:
class GeneImportanceLayer(nn.Module):
    """Per-gene learnable sigmoid gate. T-GEM inspired."""
    def __init__(self, num_genes, init_val=1.5):
        super().__init__()
        self.logits = nn.Parameter(torch.full((num_genes,), init_val))

    def forward(self, x):
        return x * torch.sigmoid(self.logits)


class ModalityEncoder(nn.Module):
    """GeneGate (optional) -> Linear -> CNN -> CLS + TransformerEncoder"""
    def __init__(self, num_genes, embed_dim, num_heads, num_layers,
                 cnn_kernel, cnn_stride, ff_dim, dropout,
                 gate_init=1.5, use_ckpt=False, use_gene_gate=True):
        super().__init__()
        self.use_ckpt = use_ckpt

        self.gene_gate = GeneImportanceLayer(num_genes, init_val=gate_init) if use_gene_gate else None

        self.input_proj = nn.Linear(1, embed_dim)

        self.cnn = nn.Sequential(
            nn.Conv1d(embed_dim, embed_dim, kernel_size=cnn_kernel, stride=cnn_stride,
                      padding=cnn_kernel // 2),
            nn.GELU(),
            nn.BatchNorm1d(embed_dim),
        )

        self.cls_token = nn.Parameter(torch.randn(1, 1, embed_dim) * 0.02)

        compressed = (num_genes + cnn_kernel // 2 * 2 - cnn_kernel) // cnn_stride + 1 + 1
        self.pos_emb = nn.Parameter(torch.randn(1, compressed, embed_dim) * 0.02)

        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim, nhead=num_heads, dim_feedforward=ff_dim, dropout=dropout,
            batch_first=True, norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)
        self.norm = nn.LayerNorm(embed_dim)

    def _tfm(self, x): return self.transformer(x)

    def forward(self, x):
        if self.gene_gate is not None:
            x = self.gene_gate(x)
        x = x.unsqueeze(-1)
        x = self.input_proj(x)
        x = x.transpose(1, 2)
        x = self.cnn(x)
        x = x.transpose(1, 2)

        B = x.size(0)
        cls = self.cls_token.expand(B, -1, -1)
        x = torch.cat([cls, x], dim=1)
        x = x + self.pos_emb[:, :x.size(1), :]

        if self.use_ckpt and self.training:
            x = grad_checkpoint(self._tfm, x, use_reentrant=False)
        else:
            x = self._tfm(x)

        return self.norm(x)


class CrossModalAttention4(nn.Module):
    """4-modality cross-modal attention."""
    def __init__(self, embed_dim, num_heads, ff_dim, dropout):
        super().__init__()
        mods = ['mrna', 'cnv', 'mut', 'meth']
        self.cross_attns = nn.ModuleDict({
            m: nn.MultiheadAttention(embed_dim, num_heads, dropout=dropout, batch_first=True)
            for m in mods
        })
        self.q_norms = nn.ModuleDict({m: nn.LayerNorm(embed_dim) for m in mods})

        def make_ffn():
            return nn.Sequential(
                nn.Linear(embed_dim, ff_dim), nn.GELU(), nn.Dropout(dropout),
                nn.Linear(ff_dim, embed_dim), nn.Dropout(dropout),
            )
        self.ffns      = nn.ModuleDict({m: make_ffn() for m in mods})
        self.ffn_norms = nn.ModuleDict({m: nn.LayerNorm(embed_dim) for m in mods})

    def forward(self, mrna_seq, cnv_seq, mut_seq, meth_seq):
        seqs = {'mrna': mrna_seq, 'cnv': cnv_seq, 'mut': mut_seq, 'meth': meth_seq}
        out  = {}
        for m, q_seq in seqs.items():
            others = torch.cat([s for k, s in seqs.items() if k != m], dim=1)
            q = self.q_norms[m](q_seq)
            h, _ = self.cross_attns[m](q, others, others, need_weights=False)
            attn_out = q_seq + h
            h2 = self.ffns[m](self.ffn_norms[m](attn_out))
            out[m] = attn_out + h2
        return out['mrna'], out['cnv'], out['mut'], out['meth']


class GatedFusion4(nn.Module):
    """4-modality gated fusion."""
    def __init__(self, embed_dim):
        super().__init__()
        self.gate = nn.Linear(embed_dim * 4, 4)

    def forward(self, cls_mrna, cls_cnv, cls_mut, cls_meth):
        concat = torch.cat([cls_mrna, cls_cnv, cls_mut, cls_meth], dim=-1)
        gates  = F.softmax(self.gate(concat), dim=-1)
        fused = (gates[:, 0:1] * cls_mrna + gates[:, 1:2] * cls_cnv +
                 gates[:, 2:3] * cls_mut  + gates[:, 3:4] * cls_meth)
        return fused, gates


class OncoBridgeMMCAT_v7(nn.Module):
    """OncoBridge Multi-Modal Cross-Attention Transformer v7 (ablation-parameterised)."""
    def __init__(self, cfg):
        super().__init__()
        E   = cfg['embed_dim']
        H   = cfg['num_heads']
        NL  = cfg['num_encoder_layers']
        NC  = cfg['num_cross_layers']
        K   = cfg['cnn_kernel']
        S   = cfg['cnn_stride']
        FF  = cfg['ff_dim']
        D   = cfg['dropout']
        GI  = cfg['gate_init']
        UC  = cfg['use_grad_ckpt']
        UGG = cfg['use_gene_gate']

        self.mrna_enc = ModalityEncoder(cfg['num_mrna_genes'], E, H, NL, K, S, FF, D, GI, UC, UGG)
        self.cnv_enc  = ModalityEncoder(cfg['num_cnv_genes'],  E, H, NL, K, S, FF, D, GI, UC, UGG)
        self.mut_enc  = ModalityEncoder(cfg['num_mut_genes'],  E, H, NL, K, S, FF, D, GI, UC, UGG)
        self.meth_enc = ModalityEncoder(cfg['num_meth_genes'], E, H, NL, K, S, FF, D, GI, UC, UGG)

        self.cross_layers = nn.ModuleList([CrossModalAttention4(E, H, FF, D) for _ in range(NC)])
        self.fusion = GatedFusion4(E)

        self.classifier = nn.Sequential(
            nn.LayerNorm(E * 5),
            nn.Linear(E * 5, E * 2), nn.GELU(), nn.Dropout(D),
            nn.Linear(E * 2, E), nn.GELU(), nn.Dropout(D),
            nn.Linear(E, cfg['num_classes']),
        )

    def forward(self, mrna, cnv, mut, meth):
        mrna_seq = self.mrna_enc(mrna)
        cnv_seq  = self.cnv_enc(cnv)
        mut_seq  = self.mut_enc(mut)
        meth_seq = self.meth_enc(meth)

        for layer in self.cross_layers:
            mrna_seq, cnv_seq, mut_seq, meth_seq = layer(mrna_seq, cnv_seq, mut_seq, meth_seq)

        cls_m = mrna_seq[:, 0]
        cls_c = cnv_seq[:,  0]
        cls_u = mut_seq[:,  0]
        cls_e = meth_seq[:, 0]

        fused, gates = self.fusion(cls_m, cls_c, cls_u, cls_e)
        combined = torch.cat([cls_m, cls_c, cls_u, cls_e, fused], dim=-1)
        return self.classifier(combined)


model = OncoBridgeMMCAT_v7(CONFIG).to(DEVICE)
if N_GPUS > 1:
    model = nn.DataParallel(model)
    print(f'DataParallel across {N_GPUS} GPUs')

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'\nModel: OncoBridgeMMCAT v7  |  Ablation: {ABLATION}')
print(f'  Parameters:      {n_params:,}')
print(f'  use_gene_gate:   {CONFIG["use_gene_gate"]}')
print(f'  num_cross_layers:{CONFIG["num_cross_layers"]}')
print(f'  num_heads:       {CONFIG["num_heads"]}')

DataParallel across 2 GPUs

Model: OncoBridgeMMCAT v7  |  Ablation: no_gene_gate
  Parameters:      82,705,188
  use_gene_gate:   False
  num_cross_layers:4
  num_heads:       8


## Loss, optimizer, scheduler — identical to the main notebook (full budget)

In [9]:
criterion = nn.CrossEntropyLoss(weight=cw_tensor, label_smoothing=CONFIG['label_smoothing'])

optimizer = optim.AdamW(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'],
                         betas=(0.9, 0.999), fused=(DEVICE.type == 'cuda'))

steps_per_epoch = max(1, math.ceil(len(train_loader) / CONFIG['grad_accum']))
total_steps     = steps_per_epoch * CONFIG['epochs']

scheduler = torch.optim.lr_scheduler.OneCycleLR(
    optimizer, max_lr=CONFIG['lr'], total_steps=total_steps,
    pct_start=CONFIG['warmup_frac'], anneal_strategy='cos',
    div_factor=10.0, final_div_factor=1000.0,
)

amp_scaler = torch.cuda.amp.GradScaler(enabled=CONFIG['use_amp'])
print(f'Optimizer: AdamW(lr={CONFIG["lr"]}) | Scheduler: OneCycleLR | '
      f'total_steps={total_steps} | patience={CONFIG["patience"]} (full budget)')

Optimizer: AdamW(lr=0.0002) | Scheduler: OneCycleLR | total_steps=1540 | patience=18 (full budget)


## Training loop — identical to the main notebook, full budget (140 epochs, patience 18)

In [10]:
def mixup(mrna, cnv, mut, meth, labels, alpha=0.2):
    dist = torch.distributions.Beta(torch.tensor(alpha, device=DEVICE), torch.tensor(alpha, device=DEVICE))
    lam  = dist.sample().item()
    perm = torch.randperm(labels.size(0), device=DEVICE)
    return (lam * mrna + (1 - lam) * mrna[perm], lam * cnv + (1 - lam) * cnv[perm],
            lam * mut + (1 - lam) * mut[perm], lam * meth + (1 - lam) * meth[perm],
            labels, labels[perm], lam)


def run_epoch(model, loader, crit, opt, amp_sc, is_train, sched=None):
    model.train() if is_train else model.eval()
    tot_loss = tot_correct = tot_n = 0
    accum = CONFIG['grad_accum'] if is_train else 1
    if is_train: opt.zero_grad(set_to_none=True)

    for step, (mrna, cnv, mut, meth, labels) in enumerate(loader):
        mrna, cnv, mut, meth, labels = (t.to(DEVICE, non_blocking=True) for t in (mrna, cnv, mut, meth, labels))

        do_mix = (is_train and CONFIG['use_mixup'] and torch.rand(1).item() < CONFIG['mixup_prob'])

        with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
            if do_mix:
                xm, xc, xu, xe, ya, yb, lam = mixup(mrna, cnv, mut, meth, labels, CONFIG['mixup_alpha'])
                out  = model(xm, xc, xu, xe)
                loss = (lam * crit(out, ya) + (1 - lam) * crit(out, yb)) / accum
            else:
                out  = model(mrna, cnv, mut, meth)
                loss = crit(out, labels) / accum

        if is_train:
            amp_sc.scale(loss).backward()
            last_step = (step + 1 == len(loader))
            if (step + 1) % accum == 0 or last_step:
                amp_sc.unscale_(opt)
                nn.utils.clip_grad_norm_(model.parameters(), CONFIG['clip_grad'])
                amp_sc.step(opt)
                amp_sc.update()
                opt.zero_grad(set_to_none=True)
                if sched is not None: sched.step()

        tot_loss += loss.item() * accum
        preds = out.argmax(1)
        if do_mix:
            soft = lam * preds.eq(ya).float() + (1 - lam) * preds.eq(yb).float()
            tot_correct += soft.sum().item()
        else:
            tot_correct += preds.eq(labels).sum().item()
        tot_n += labels.size(0)

    return tot_loss / len(loader), 100.0 * tot_correct / tot_n


history = {k: [] for k in ['train_loss', 'train_acc', 'val_loss', 'val_acc']}
best_val_acc, patience_cnt, best_state = 0.0, 0, None

print('=' * 100)
print(f'  OncoBridge-MMCAT v7  |  Ablation: {ABLATION}  |  Full budget ({CONFIG["epochs"]} epochs, patience {CONFIG["patience"]})')
print('=' * 100)

for epoch in range(1, CONFIG['epochs'] + 1):
    tr_loss, tr_acc = run_epoch(model, train_loader, criterion, optimizer, amp_scaler, True, scheduler)
    with torch.no_grad():
        vl_loss, vl_acc = run_epoch(model, val_loader, criterion, optimizer, amp_scaler, False)

    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)

    if vl_acc > best_val_acc:
        best_val_acc = vl_acc
        patience_cnt = 0
        best_state = copy.deepcopy(model.state_dict())
        torch.save(best_state, CONFIG['checkpoint'])
        status = 'BEST'
    else:
        patience_cnt += 1
        status = f'({patience_cnt}/{CONFIG["patience"]})'

    lr = optimizer.param_groups[0]['lr']
    print(f'{epoch:>4} | lr={lr:.6f} | tr_loss={tr_loss:.4f} tr_acc={tr_acc:.2f}% | '
          f'vl_loss={vl_loss:.4f} vl_acc={vl_acc:.2f}% | {status}')

    if patience_cnt >= CONFIG['patience']:
        print(f'\nEarly stop at epoch {epoch}. Best val: {best_val_acc:.2f}%')
        break

print(f'\nTraining complete [{ABLATION}]. Best validation accuracy: {best_val_acc:.2f}%')

  OncoBridge-MMCAT v7  |  Ablation: no_gene_gate  |  Full budget (140 epochs, patience 18)
   1 | lr=0.000021 | tr_loss=3.6205 tr_acc=10.74% | vl_loss=3.5624 vl_acc=17.27% | BEST
   2 | lr=0.000024 | tr_loss=3.4870 tr_acc=19.22% | vl_loss=3.4290 vl_acc=23.50% | BEST
   3 | lr=0.000029 | tr_loss=3.3288 tr_acc=24.23% | vl_loss=3.2507 vl_acc=28.56% | BEST
   4 | lr=0.000036 | tr_loss=3.1486 tr_acc=29.55% | vl_loss=3.0747 vl_acc=28.73% | BEST
   5 | lr=0.000044 | tr_loss=2.9207 tr_acc=33.35% | vl_loss=2.8021 vl_acc=35.30% | BEST
   6 | lr=0.000054 | tr_loss=2.6879 tr_acc=37.20% | vl_loss=2.8659 vl_acc=26.03% | (1/18)
   7 | lr=0.000065 | tr_loss=2.5342 tr_acc=40.28% | vl_loss=2.6239 vl_acc=35.13% | (2/18)
   8 | lr=0.000078 | tr_loss=2.2650 tr_acc=47.62% | vl_loss=2.4365 vl_acc=38.08% | BEST
   9 | lr=0.000090 | tr_loss=2.2772 tr_acc=45.41% | vl_loss=2.5123 vl_acc=38.42% | BEST
  10 | lr=0.000104 | tr_loss=2.1982 tr_acc=48.99% | vl_loss=2.7991 vl_acc=27.04% | (1/18)
  11 | lr=0.000117 | tr

## Evaluation and results logging

In [11]:
def full_eval(loader, split_name):
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_labels = [], []
    with torch.no_grad():
        for mrna, cnv, mut, meth, labels in loader:
            mrna, cnv, mut, meth = (t.to(DEVICE, non_blocking=True) for t in (mrna, cnv, mut, meth))
            with torch.cuda.amp.autocast(enabled=CONFIG['use_amp']):
                out = model(mrna, cnv, mut, meth)
            all_preds.extend(out.argmax(1).cpu().numpy())
            all_labels.extend(labels.numpy())

    preds  = np.array(all_preds)
    labels = np.array(all_labels)
    acc = accuracy_score(labels, preds) * 100
    wf1 = f1_score(labels, preds, average='weighted')
    mf1 = f1_score(labels, preds, average='macro')
    mcc = matthews_corrcoef(labels, preds)

    print(f'\n{"="*60}\n  {split_name} Results [{ABLATION}]\n{"="*60}')
    print(f'  Accuracy:    {acc:.2f}%')
    print(f'  Weighted F1: {wf1:.4f}')
    print(f'  Macro F1:    {mf1:.4f}')
    print(f'  MCC:         {mcc:.4f}')
    return acc, wf1, mf1, mcc

val_acc_final, val_wf1, val_mf1, val_mcc = full_eval(val_loader, 'Validation')
if test_loader is not None:
    te_acc, te_wf1, te_mf1, te_mcc = full_eval(test_loader, 'Test (held-out)')
else:
    te_acc, te_wf1, te_mf1, te_mcc = (None,) * 4

# ── Append this run's result to the shared results CSV ─────────────────────
row = {
    'ablation':        ABLATION,
    'test_accuracy':   te_acc,
    'test_weighted_f1':te_wf1,
    'test_macro_f1':   te_mf1,
    'test_mcc':        te_mcc,
    'val_accuracy':    val_acc_final,
    'epochs_trained':  len(history['train_acc']),
    'checkpoint':      CONFIG['checkpoint'],
    'timestamp':       datetime.now().isoformat(timespec='seconds'),
}

file_exists = os.path.isfile(RESULTS_LOG)
with open(RESULTS_LOG, 'a', newline='') as f:
    writer = csv.DictWriter(f, fieldnames=list(row.keys()))
    if not file_exists:
        writer.writeheader()
    writer.writerow(row)

print(f'\nAppended results for "{ABLATION}" to {RESULTS_LOG}')
print('\nCurrent results log:')
print(pd.read_csv(RESULTS_LOG).to_string(index=False))


  Validation Results [no_gene_gate]
  Accuracy:    92.59%
  Weighted F1: 0.9237
  Macro F1:    0.8816
  MCC:         0.9223

  Test (held-out) Results [no_gene_gate]
  Accuracy:    91.66%
  Weighted F1: 0.9140
  Macro F1:    0.8671
  MCC:         0.9127

Appended results for "no_gene_gate" to /kaggle/working/oncobridge_s14_full_convergence_results.csv

Current results log:
    ablation  test_accuracy  test_weighted_f1  test_macro_f1  test_mcc  val_accuracy  epochs_trained                              checkpoint           timestamp
no_gene_gate      91.659646          0.913983       0.867127  0.912668     92.586352             121 oncobridge_s14_no_gene_gate_fullconv.pt 2026-07-13T20:33:30


## Next steps

1. Note the test-set numbers printed above for this ablation.
2. Change `ABLATION` in the config cell near the top to the next option (`'shallow_cross_attn'`, `'few_heads'`, or `'no_gene_gate'` — whichever hasn't run yet), then **Restart & Run All**.
3. Once all three (or four, including `'full'`) have run, `RESULTS_LOG` will contain one row per ablation with directly comparable test-set accuracy, weighted F1, macro F1, and MCC — all at the same full training budget as the 95.74% reference, so these numbers can replace the reduced-budget S1.4 sweep in Table 6c.
